# M01C05: GIL and Reference Counting; Cycle GC, `weakref`, Finalizers

**System Information:** Apple M1 Max (macOS), Python 3.14.0

**Version Diagnostic:**

```python:disable-run
import sys
print(f"CPython {sys.version} ({sys.implementation.cache_tag})")
assert sys.implementation.name == 'cpython', "This module assumes CPython 3.12+; adjust for alternatives."
```

> **Guidance for Extension Developers:** This module details CPython's GIL-refcount interplay and cycle detection. For portable extensions, avoid `__del__` reliance; use `weakref` compatibly via stable C API (PEP 384) or HPy. Consult [HPy documentation](https://hpyproject.org/) for thread-safe patterns.

This module examines CPython's primary deallocation mechanism—reference counting under the GIL—and its limitations (cycles), augmented by generational GC. Extending M01C01–C04's model, we explore `weakref` for cycle avoidance and finalizers for cleanup, with implications for thread-safety, leaks, and free-threaded futures (PEP 703). Explanations integrate empirical labs, historical context (e.g., GIL's ABC roots), and edge cases, ensuring self-contained precision.

> **Fact Typing Convention:** Throughout this module, facts are classified as follows for clarity and longevity:
> - **Semantic Invariant:** Portable across Python implementations (e.g., PyPy); enshrined in the language specification.
> - **CPython Implementation Detail:** Specific to CPython 3.12–3.14; measured empirically; subject to evolution without notice.
> - **Forward-Looking Note:** Based on PEPs (e.g., 703); provisional until stabilized.

Structure: Motivational labs, GIL-refcount duet, cycle pitfalls, GC augmentation, `weakref`/finalizers, verification, and synthesis. All code executes in CPython 3.12+.

## Glossary of Key Terms

For quick reference, the following table defines abbreviations and concepts used throughout. Expand for full details.

<details>
<summary>Expand Glossary</summary>

| Term/Abbreviation | Definition |
|-------------------|------------|
| **GIL** | Global Interpreter Lock: Serializes bytecode execution in CPython, ensuring thread-safe refcount ops without locks. |
| **Reference Counting** | Primary deallocation: `ob_refcnt` INC/DEC on ref creation/drop; zero triggers free (cycles excepted). |
| **Cycle GC** | Generational collector augmenting refcounting: Detects/traverses mutual refs in containers (gen 0–2). |
| **weakref** | Non-strong reference: Doesn't INC refcnt; enables cycle-breaking back-edges (e.g., parent ptrs). |
| **`__del__`** | Dunder method for finalization: Unreliable in cycles/shutdown; avoid for critical cleanup. |
| **`weakref.finalize`** | Best-effort callback on deallocation: Safer than `__del__`; no cycle retention. |
| **Generation** | GC age bucket: New objects gen 0 (frequent scans); survivors promote to gen 1/2 (rarer). |
| **PEP 703** | Free-threaded CPython: GIL removal; refcounts atomic/deferred, altering concurrency costs. |

</details>

## 0. Why You Care: Motivational Experiments

Refcounting enables deterministic frees but falters on cycles; GIL ensures thread-safety sans locks. These labs—cycle reclamation and concurrent refcounts—motivate the duet below, underscoring why `weakref` is essential for graphs.

**Scope:** **CPython 3.12+** 64-bit release (GIL-present); free-threaded (PEP 703) uses atomics. Behaviors spec-aligned, but details (e.g., gen promotions) CPython-specific.

**Caveats:**
- Portable: Cycle semantics standard; GIL CPython-only.
- Extensions: `Py_INCREF`/`Py_DECREF` GIL-safe; HPy for portables.

**Version Matrix:** *(CPython Implementation Detail)*

| Version | GC Gens | GIL Status | Key Notes |
|---------|---------|------------|-----------|
| 3.12   | 0–2    | Present   | Baseline refcount. |
| 3.13   | 0–2    | Experimental free-threaded | Atomics optional. |
| 3.14   | 0–2    | Stabilized free-threading | Deferred refcounts provisional. |

**Guiding Questions:**
- Why do cycles evade refcounting?
- How does GIL prevent refcount races?
- When to prefer `weakref` over strong?

### Experiment 1: Cycles Break Pure Refcounting

Mutual refs hold refcnt=1 post-del; GC reclaims via traversal. *(Semantic Invariant: Cycles leak sans GC; portable.)*

In [1]:
import gc
import weakref

class C:
    def __init__(self):
        self.other: C | None = None

def make_cycle():
    a = C()
    b = C()
    a.other = b
    b.other = a
    return weakref.ref(a)

wr = make_cycle()
assert wr() is not None

gc.collect()
print(f"Cycle reclaimed? {wr() is None}")  # True

# Expected: GC breaks cycle; weakref confirms free.

Cycle reclaimed? True


### Experiment 2: Refcounting Thread-Safe via GIL

Concurrent INC/DEC; no corruption. *(CPython Implementation Detail: GIL serializes.)*

In [2]:
import threading
import sys

obj = object()

def inc_ref(n):
    global obj
    for _ in range(n):
        ref = obj  # +1
        # Scope out → -1

threads = [threading.Thread(target=inc_ref, args=(10_000_000,)) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Final refcount: {sys.getrefcount(obj)}")  # Sane (2)

# Expected: No race; GIL protects.

Final refcount: 2


## 1. Scope Reminder

**CPython 3.12+** non-free-threaded (GIL-present); free-threaded experimental (atomics/deferred). Internals evolve; spec (e.g., weakref semantics) stable.

## 2. The GIL + Reference Counting Duet

**Semantic Invariant:** Refcounting: `ob_refcnt` INC/DEC per ref; zero → free (cycles excepted).

**CPython Implementation Detail:** GIL serializes bytecode: Thread-safe INC/DEC sans locks (~O(1) deref). No torn reads; but locks needed for multi-op invariants (e.g., CAS).

**Consequences:**
| Aspect | Under GIL | Free-Threaded (Forward-Looking Note) |
|--------|-----------|-------------------------------------|
| Refcount Safety | Serialized; lock-free | Atomics/deferred; ~10–20% overhead |
| Parallelism | I/O-bound only | CPU-bound viable |
| GC Pauses | All threads | Per-gen, concurrent potential |

**Rationale:** GIL (ABC heritage) prioritizes single-thread perf; PEP 703 trades for parallelism.

## 3. Reference Counting in Practice

`sys.getrefcount` +1 temporary; deltas exact. *(CPython Implementation Detail: ob_refcnt signed ssize_t.)*

In [3]:
import sys

obj = object()
print(sys.getrefcount(obj))  # ≥2

tmp = obj
print(sys.getrefcount(obj))  # +1

del tmp
print(sys.getrefcount(obj))  # Back

# Lab 1: Thread-visible immediacy
import threading

def worker():
    print(f"Worker start: {sys.getrefcount(obj)}")  # +1 arg
    tmp = obj
    print(f"Worker peak: {sys.getrefcount(obj)}")   # +1 more
    del tmp
    print(f"Worker end: {sys.getrefcount(obj)}")

t = threading.Thread(target=worker)
t.start()
t.join()

# Expected: Consistent; GIL orders.

2
3
2
Worker start: 2
Worker peak: 3
Worker end: 2


## 4. The Cycle Problem (GIL Irrelevant)

Mutual refs: refcnt=1 post-del; leak forever. Common in trees (parents), caches, observers. *(Semantic Invariant: Cycles evade refcount; portable leak risk.)*

In [4]:
class C:
    def __init__(self):
        self.other = None

a = C()
b = C()
a.other = b
b.other = a  # refcnt=2 each

del a, b  # refcnt=1 each; leak

# Insight: Back-edges trap; GIL doesn't detect.

## 5. Generational Cycle GC: The Safety Net

**CPython Implementation Detail:** Tracks containers (lists/dicts/sets/classes); gens 0–2: New → gen0 (frequent); survivors promote (rarer scans). Thresholds: gen0 every ~700 containers; gen1/2 rarer.

**Algorithm (Simplified):**
1. Subtract internal refs temporarily.
2. Zero adjusted refcnt → garbage.
3. Free; restore survivors.

Runs with GIL (pauses threads); ~1–5% overhead in cycle-prone code. *(Forward-Looking Note: PEP 703 concurrent GC potential.)*

**Lab 2: Cycle GC in Action**

List cycle; weakref probes free. *(Semantic Invariant: GC reclaims cycles; portable.)*

In [5]:
import gc
import weakref

class C:
    def __init__(self, val):
        self.val = val
        self.other = None  # Mutable attribute to form the cycle

def make_cycle():
    a = C(1)
    b = C(2)
    a.other = b  # Strong reference
    b.other = a  # Strong reference, creating mutual cycle
    return weakref.ref(a)  # Weak reference to probe survival

wr = make_cycle()
gc.collect()
print(f"Cycle reclaimed? {wr() is None}")  # Expected: True

Cycle reclaimed? True


## 6. `weakref`: The Escape Hatch

Non-INC refs: Break cycles. Ideal for parents/observers/caches. *(Semantic Invariant: Weakrefs don't retain; portable semantics.)*

In [6]:
import weakref

class Node:
    def __init__(self, value):
        self.value = value
        self.parent = None  # Weak

    def set_parent(self, parent):
        self.parent = weakref.ref(parent)

root = Node("root")
child = Node("child")
child.set_parent(root)

wr = weakref.ref(child)
del root, child
gc.collect()
print(f"Tree reclaimed? {wr() is None}")  # True

# Insight: No cycle; weak frees.

Tree reclaimed? True


## 7. Finalizers: Avoid `__del__`, Use `weakref.finalize`

`__del__` unreliable (cycles/shutdown); `finalize` best-effort, cycle-safe. *(CPython Implementation Detail: `__del__` suppressed in GC.)*

In [7]:
import weakref
import gc

def test_finalizer():
    class MyObject:
        """Simple user-defined class supporting weak references."""
        pass
    
    obj = MyObject()
    finalizer = weakref.finalize(obj, lambda: print("goodbye"))
    
    del obj  # Reduces refcount to zero
    gc.collect()  # Forces deallocation and callback invocation
    
    assert not finalizer.alive  # Passes: finalizer is inactive

test_finalizer()

goodbye


## 8. Verification: Core Invariants

Asserts: Cycle reclaim, weak breaks, thread-safe refcount. *(CPython Implementation Detail: Tune loops for env.)*

In [8]:
import gc
import weakref
import threading
import sys

def test_cycle_is_reclaimed():
    class C:
        def __init__(self): self.other = None
    a = C(); b = C(); a.other = b; b.other = a
    wr = weakref.ref(a)
    del a, b
    gc.collect()
    assert wr() is None

def test_weakref_breaks_cycle():
    class Node:
        def set_parent(self, p): self.parent = weakref.ref(p)
    root = Node(); child = Node(); child.set_parent(root)
    root.child = child
    wr = weakref.ref(child)
    del root, child
    gc.collect()
    assert wr() is None

def test_refcount_thread_safe():
    obj = object()
    base = sys.getrefcount(obj)
    def bump():
        for _ in range(100_000): tmp = obj
    threads = [threading.Thread(target=bump) for _ in range(10)]
    for t in threads: t.start()
    for t in threads: t.join()
    assert sys.getrefcount(obj) == base

# Run
suite = [test_cycle_is_reclaimed, test_weakref_breaks_cycle, test_refcount_thread_safe]
for test in suite:
    test()
print("\nSuite: Lifetime invariants hold.")


Suite: Lifetime invariants hold.


**Exercise (Mandatory):** Doubly-linked list/tree with parents; induce cycles; convert back/parents to weakref. Prove reclaim/thread-safety via weakref + GC + threads. *(Semantic Invariant: Weak enables acyclic graphs; portable.)*

## 9. Essential Takeaways: Invariants to Internalize

| Invariant | Implication | Mitigation |
|-----------|-------------|------------|
| Refcount + GIL | Fast, thread-safe INC/DEC | Locks for multi-op |
| Cycles Leak | Back-edges trap | weakref for parents/caches |
| Gen GC Augments | ~1–5% overhead; pauses | Threshold tuning (gc.set_threshold) |
| __del__ Unreliable | Cycles/shutdown suppress | weakref.finalize best-effort |
| Free-Threaded (Forward-Looking Note) | Atomics overhead | Benchmark separately |

**Synthesis:** Refcounting deterministic; GC safety-net; weak composes safe graphs (M01C02).

## 10. Progression: Subsequent Modules in M01

Lifetimes mastered; measure next:

- **M01C06: Advanced Measurement** – Footprint profiling.
- **M01C07+: Layouts, Slots, Descriptors** – Allocation ties.

Invariants fortify graph reasoning.

## Appendix: Sample Exercise Solution

Doubly-linked with weak backs; cycle test; thread-safe.

In [9]:
import gc
import weakref
import threading
import sys

class DNode:
    def __init__(self, value):
        self.value = value
        self.prev = None  # Weak
        self.next = None  # Weak

    def set_prev(self, prev): self.prev = weakref.ref(prev)
    def set_next(self, next): self.next = weakref.ref(next)

# Build cycle
a = DNode('a'); b = DNode('b')
a.set_next(b); b.set_prev(a)
b.set_next(a); a.set_prev(b)

wr_a = weakref.ref(a)
del a, b
gc.collect()
assert wr_a() is None  # Reclaimed

# Thread-safe: Concurrent access
def access(node):
    for _ in range(100_000):
        tmp = node.value

t = threading.Thread(target=access, args=(DNode('safe'),))
t.start(); t.join()

print("Solution: Cycles broken; safe.")

# Insight: Weak bidirectional; no trap.

Solution: Cycles broken; safe.
